# SPPACEU — 1

Run from the repository directory after installing `requirements.txt`. Configuration uses environment variables described in README.md. External processing is disabled by default. Choose one video-screening variant (steps 3–7); do not run all variants against the same outputs.


In [ ]:
import os
from pathlib import Path
from notebook_runtime import DATA_DIR, RECORDS_DIR, MODEL_NAME, load_openai_key, prepare_run

RUN_ANALYSIS = False  # Enable explicitly after reviewing inputs and API/download settings.
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import time
from pathlib import Path
from typing import Optional, Dict, Any, List
import pandas as pd
from openai import OpenAI

In [ ]:
# =========================
# CONFIGURATION
# =========================
INPUT_XLSX = DATA_DIR / "youtube_data.xlsx"
OUTPUT_XLSX = DATA_DIR / "channel_ai_analysis.xlsx"
LOG_XLSX = DATA_DIR / "channel_ai_processing_log.xlsx"
API_KEY_PATH = None  # Read OPENAI_API_KEY from the environment.
SHEET_CHANNEL = "channel"
MODEL_NAME = os.environ.get("OPENAI_MODEL", "").strip()
MAX_RETRIES = 3
REQUEST_TIMEOUT = 180
SLEEP_BETWEEN_REQUESTS = 1.0
SKIP_EXISTING = True
# If you want to test only first N rows, set e.g. 10.
# If None, process all rows.
LIMIT_ROWS: Optional[int] = None

In [ ]:
# =========================

# EXPECTED COLUMNS

# =========================

REQUIRED_COLUMNS = [

    "channel_title",

    "description",

]

OPTIONAL_COLUMNS = [

    "channel_id",

    "channel_url",

    "country",

    "view_count",

    "subscriber_count",

    "video_count",

    "published_at",

]

# =========================

# SYSTEM PROMPT

# =========================

SYSTEM_INSTRUCTION = """

You are analyzing YouTube channel metadata for a sociological research project.

Research topic:

"Social practices of public audit and corruption exposure in Ukraine."

Your task is to classify a YouTube channel based only on the provided metadata:

- channel title;

- channel description;

- country, if available;

- subscriber count, video count, view count, publication date, if available.

Do not invent facts. Do not use external knowledge about the channel.

Classify only what is supported by the metadata.

Return ONLY valid JSON. No markdown. No explanatory text outside JSON.

Use this JSON schema:

{

  "channel_topic": "string",

  "secondary_topics": ["string"],

  "channel_function": "string",

  "channel_type": "string",

  "corruption_relevance_category": "string",

  "public_audit_relevance_category": "string",

  "ukraine_relevance_category": "string",

  "overall_relevance_score": 0.0,

  "classification_confidence": 0.0,

  "inclusion_status": "string",

  "manual_review_required": true,

  "evidence_markers": ["string"],

  "risk_notes": ["string"],

  "short_rationale": "string"

}

Definitions:

channel_topic:

Main thematic field of the channel.

secondary_topics:

Additional thematic fields if visible.

channel_function:

One of:

- investigative

- analytical

- news

- civic_activism

- governmental_or_institutional

- educational

- commentary

- entertainment

- promotional

- mixed

- unclear

channel_type:

One of:

- individual_author

- media_project

- NGO_or_civic_project

- governmental_body

- political_actor

- educational_project

- commercial_project

- aggregator

- mixed_or_unclear

corruption_relevance_category:

One of:

- direct_corruption_focus

- indirect_governance_or_accountability_focus

- possible_contextual_relevance

- low_relevance

- unclear

public_audit_relevance_category:

One of:

- direct_public_audit_focus

- civic_monitoring_or_watchdog_focus

- transparency_or_accountability_focus

- possible_contextual_relevance

- low_relevance

- unclear

ukraine_relevance_category:

One of:

- direct_ukraine_focus

- ukrainian_language_or_context

- possible_ukraine_context

- no_visible_ukraine_focus

- unclear

overall_relevance_score:

A number from 0.00 to 1.00 estimating relevance to the research topic.

Use:

0.80-1.00 = highly relevant

0.60-0.79 = relevant but requires review

0.40-0.59 = partially/contextually relevant

0.00-0.39 = low relevance

classification_confidence:

A number from 0.00 to 1.00 estimating confidence based on available metadata.

inclusion_status:

One of:

- include_core

- include_after_manual_review

- contextual_or_borderline

- exclude

- insufficient_data

manual_review_required:

true if metadata is ambiguous, incomplete, contradictory, or relevance is borderline.

evidence_markers:

Short words or phrases from the title/description that support the classification.

risk_notes:

Possible limitations, such as empty description, branding-only title, ambiguous political language, promotional framing, missing country, etc.

short_rationale:

2-4 sentences explaining the classification in academic language.

""".strip()

# =========================

# HELPERS

# =========================

def load_api_key() -> str:
    return load_openai_key(API_KEY_PATH)


def safe_value(row: pd.Series, col: str) -> str:

    if col not in row.index:

        return ""

    val = row[col]

    if pd.isna(val):

        return ""

    return str(val).strip()

def build_user_prompt(row: pd.Series) -> str:

    data = {

        "channel_id": safe_value(row, "channel_id"),

        "channel_url": safe_value(row, "channel_url"),

        "channel_title": safe_value(row, "channel_title"),

        "description": safe_value(row, "description"),

        "country": safe_value(row, "country"),

        "view_count": safe_value(row, "view_count"),

        "subscriber_count": safe_value(row, "subscriber_count"),

        "video_count": safe_value(row, "video_count"),

        "published_at": safe_value(row, "published_at"),

    }

    return (

        "Classify the following YouTube channel metadata for the research topic:\n"

        "\"Social practices of public audit and corruption exposure in Ukraine.\"\n\n"

        "CHANNEL METADATA JSON:\n"

        f"{json.dumps(data, ensure_ascii=False, indent=2)}"

    )

def parse_json_response(text: str) -> Dict[str, Any]:

    text = text.strip()

    # Defensive cleanup if model accidentally wraps JSON.

    if text.startswith("```"):

        text = text.strip("`")

        if text.lower().startswith("json"):

            text = text[4:].strip()

    return json.loads(text)

def call_model(client: OpenAI, row: pd.Series) -> Dict[str, Any]:

    user_prompt = build_user_prompt(row)

    response = client.chat.completions.create(

        model=MODEL_NAME,

        messages=[

            {"role": "system", "content": SYSTEM_INSTRUCTION},

            {"role": "user", "content": user_prompt},

        ],

        timeout=REQUEST_TIMEOUT,

    )

    content = response.choices[0].message.content

    if not content or not content.strip():

        raise ValueError("Model returned empty response.")

    return parse_json_response(content)

def validate_input(df: pd.DataFrame) -> None:

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]

    if missing:

        raise ValueError(f"Missing required columns in sheet '{SHEET_CHANNEL}': {missing}")

def make_row_key(row: pd.Series) -> str:

    channel_id = safe_value(row, "channel_id")

    if channel_id:

        return channel_id

    channel_url = safe_value(row, "channel_url")

    if channel_url:

        return channel_url

    return safe_value(row, "channel_title")

def load_existing_results() -> pd.DataFrame:

    if not OUTPUT_XLSX.exists():

        return pd.DataFrame()

    try:

        return pd.read_excel(OUTPUT_XLSX, sheet_name="channel_ai_analysis")

    except Exception:

        return pd.DataFrame()

def existing_keys(existing_df: pd.DataFrame) -> set:

    if existing_df.empty or "row_key" not in existing_df.columns:

        return set()

    return set(existing_df["row_key"].dropna().astype(str).tolist())

def save_outputs(results: List[Dict[str, Any]], log_rows: List[Dict[str, Any]]) -> None:

    results_df = pd.DataFrame(results)

    log_df = pd.DataFrame(log_rows)

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:

        results_df.to_excel(writer, index=False, sheet_name="channel_ai_analysis")

    log_df.to_excel(LOG_XLSX, index=False)

def main() -> None:

    api_key = load_api_key()

    client = OpenAI(api_key=api_key)

    if not INPUT_XLSX.exists():

        raise FileNotFoundError(f"Input Excel file not found: {INPUT_XLSX}")

    df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_CHANNEL)

    validate_input(df)

    if LIMIT_ROWS is not None:

        df = df.head(LIMIT_ROWS).copy()

    existing_df = load_existing_results()

    processed_keys = existing_keys(existing_df) if SKIP_EXISTING else set()

    results: List[Dict[str, Any]] = []

    log_rows: List[Dict[str, Any]] = []

    if not existing_df.empty and SKIP_EXISTING:

        results.extend(existing_df.to_dict(orient="records"))

    print(f"Loaded {len(df)} channel rows from sheet '{SHEET_CHANNEL}'.")

    print(f"Already processed: {len(processed_keys)}")

    print(f"Output: {OUTPUT_XLSX}")

    for idx, row in df.iterrows():

        row_key = make_row_key(row)

        if not row_key:

            log_rows.append({

                "row_index": idx,

                "row_key": "",

                "channel_title": safe_value(row, "channel_title"),

                "status": "failed",

                "error": "Cannot construct row_key: missing channel_id/channel_url/channel_title",

            })

            continue

        if SKIP_EXISTING and row_key in processed_keys:

            print(f"Skipping existing: {row_key}")

            continue

        print(f"Processing row {idx}: {safe_value(row, 'channel_title')}")

        last_error: Optional[Exception] = None

        for attempt in range(1, MAX_RETRIES + 1):

            try:

                ai = call_model(client, row)

                result_row = {

                    "row_key": row_key,

                    "source_row_index": idx,

                    "channel_id": safe_value(row, "channel_id"),

                    "channel_url": safe_value(row, "channel_url"),

                    "channel_title": safe_value(row, "channel_title"),

                    "description": safe_value(row, "description"),

                    "country": safe_value(row, "country"),

                    "view_count": safe_value(row, "view_count"),

                    "subscriber_count": safe_value(row, "subscriber_count"),

                    "video_count": safe_value(row, "video_count"),

                    "published_at": safe_value(row, "published_at"),

                    "channel_topic": ai.get("channel_topic", ""),

                    "secondary_topics": " | ".join(ai.get("secondary_topics", [])) if isinstance(ai.get("secondary_topics", []), list) else ai.get("secondary_topics", ""),

                    "channel_function": ai.get("channel_function", ""),

                    "channel_type": ai.get("channel_type", ""),

                    "corruption_relevance_category": ai.get("corruption_relevance_category", ""),

                    "public_audit_relevance_category": ai.get("public_audit_relevance_category", ""),

                    "ukraine_relevance_category": ai.get("ukraine_relevance_category", ""),

                    "overall_relevance_score": ai.get("overall_relevance_score", ""),

                    "classification_confidence": ai.get("classification_confidence", ""),

                    "inclusion_status": ai.get("inclusion_status", ""),

                    "manual_review_required": ai.get("manual_review_required", ""),

                    "evidence_markers": " | ".join(ai.get("evidence_markers", [])) if isinstance(ai.get("evidence_markers", []), list) else ai.get("evidence_markers", ""),

                    "risk_notes": " | ".join(ai.get("risk_notes", [])) if isinstance(ai.get("risk_notes", []), list) else ai.get("risk_notes", ""),

                    "short_rationale": ai.get("short_rationale", ""),

                }

                results.append(result_row)

                log_rows.append({

                    "row_index": idx,

                    "row_key": row_key,

                    "channel_title": safe_value(row, "channel_title"),

                    "status": "ok",

                    "error": "",

                })

                last_error = None

                break

            except Exception as exc:

                last_error = exc

                wait_seconds = attempt * 3

                print(f"  Attempt {attempt} failed: {exc}")

                if attempt < MAX_RETRIES:

                    print(f"  Retrying in {wait_seconds} seconds...")

                    time.sleep(wait_seconds)

        if last_error is not None:

            log_rows.append({

                "row_index": idx,

                "row_key": row_key,

                "channel_title": safe_value(row, "channel_title"),

                "status": "failed",

                "error": str(last_error),

            })

        save_outputs(results, log_rows)

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    print("Done.")

    print(f"Results saved to: {OUTPUT_XLSX}")

    print(f"Log saved to: {LOG_XLSX}")

if __name__ == "__main__" and RUN_ANALYSIS:
    prepare_run(MODEL_NAME, OUTPUT_XLSX, LOG_XLSX)
    main()
elif __name__ == "__main__":
    print("Preview mode: review configuration, then set RUN_ANALYSIS = True to execute this stage.")
